#  IA para Redes de Suministro 

👤 **Autor:** John Leonardo Vargas Mesa  
🔗 [LinkedIn](https://www.linkedin.com/in/leonardovargas/) | [GitHub](https://github.com/LeStark)  

## 📂 Repositorio en GitHub  
- 📓 **Notebooks:** [Acceder aquí](https://github.com/LeStark/Cursos/tree/main/02%20-%20IA4SC)  
- 📑 **Data sets:** [Acceder aquí](https://github.com/LeStark/Cursos/tree/main/00%20-%20Data/02%20-%20SC)  
---

# 🚛 Notebook 8 – Ruteo de Vehículos con Metaheurísticas

En este notebook desarrollamos un modelo de **optimización del ruteo de vehículos (VRP)** utilizando **metaheurísticas aplicadas sobre la red vial real de Bogotá**.  
A diferencia de los ejercicios anteriores con datos sintéticos o matrices de distancia predefinidas, aquí se emplean **rutas obtenidas directamente de OpenStreetMap (OSM)**, lo que permite simular escenarios logísticos más cercanos a la realidad.

### 🗺 Contexto del problema

El objetivo del VRP es determinar las rutas óptimas que deben seguir uno o varios vehículos para atender un conjunto de puntos de entrega, minimizando la distancia total recorrida o el tiempo de viaje, y cumpliendo con posibles restricciones operativas (capacidad, zonas de entrega, número de vehículos, etc.).

En este caso, se considera un **depósito central en la Zona Franca de Fontibón** y un conjunto de **puntos de entrega generados aleatoriamente dentro de los límites urbanos de Bogotá**.  
El modelo trabaja sobre la **malla vial real**, de modo que las distancias no son euclidianas, sino basadas en las trayectorias reales de la red de calles.

### 🎯 Objetivos del Notebook

- Construir la red vial de Bogotá a partir de datos de **OpenStreetMap** mediante la librería **osmnx**.  
- Generar y visualizar el **depósito y los puntos de entrega** en un mapa interactivo.  
- Calcular la **matriz de distancias reales** entre todos los puntos utilizando las rutas más cortas del grafo vial.  
- Implementar un **algoritmo genético (GA)** con la librería `scikit-opt` para resolver el problema de ruteo.  
- Registrar el **historial de evolución de las soluciones** (fitness y rutas) a lo largo de las generaciones.  
- Visualizar los resultados:  
  - Evolución del fitness (distancia mínima).  
  - Ruta óptima sobre el mapa de Bogotá.  

### 🧠 Metodología

1. **Definición del entorno:** descarga de la red vial con `osmnx` (modo “drive”) y creación del grafo dirigido.  
2. **Geocodificación:** definición del punto de partida (depósito) y generación de clientes dentro del polígono urbano.  
3. **Cálculo de la matriz de distancias:** uso de `networkx.shortest_path_length()` ponderado por longitud real.  
4. **Optimización:** aplicación de un **algoritmo genético** para minimizar la distancia total de recorrido.  
5. **Seguimiento del proceso evolutivo:** registro del mejor individuo y fitness por generación.  
6. **Visualización:**  
   - Gráficos de convergencia (fitness vs. generaciones).  
   - Mapa interactivo con la ruta óptima.  


### 🛠️ Herramientas utilizadas

- **osmnx / networkx:** descarga y manipulación de grafos viales.  
- **folium:** visualización interactiva sobre mapa real.  
- **scikit-opt:** implementación de metaheurísticas (GA, PSO, SA) para TSP y VRP.  
- **numpy / matplotlib / pandas:** manipulación, análisis y visualización de resultados.  

### ✅ Resultados esperados

Al finalizar el notebook, el estudiante:
- Comprende cómo integrar información geográfica real con algoritmos de optimización evolutiva.  
- Interpreta la **convergencia del GA** y la calidad de las soluciones obtenidas.  
- Visualiza rutas óptimas realistas sobre la malla vial de Bogotá.  

Este ejercicio combina **geointeligencia y optimización metaheurística**, sentando las bases para problemas de logística urbana, última milla y transporte inteligente.


In [1]:
#!pip install osmnx networkx folium 

In [2]:
# --- Librerías para manejo y obtención de datos geográficos ---
import osmnx as ox            # descarga de redes viales desde OpenStreetMap
import networkx as nx         # análisis de grafos (rutas, distancias)
from shapely.geometry import Point  # manejo de coordenadas geográficas

# --- Librerías para visualización ---
import folium                 # mapas interactivos sobre la red vial
import matplotlib.pyplot as plt  # gráficos de evolución y resultados

# --- Librerías para optimización y metaheurísticas ---
from sko.GA import GA_TSP     # algoritmo genético para resolver el TSP (base del VRP)

# --- Librerías auxiliares ---
import numpy as np            # operaciones numéricas
import random                 # generación de puntos aleatorios (clientes)
import pandas as pd           # manejo de datos en tablas (resultados)  

### Descarga de la red vial con OSMnx

En esta celda se utiliza la librería **OSMnx** para obtener la red vial de la ciudad de Bogotá directamente desde **OpenStreetMap (OSM)**.  
El comando `ox.graph_from_place()` permite crear un **grafo dirigido** donde los **nodos** representan intersecciones o puntos geográficos y las **aristas** representan segmentos de vías con atributos como distancia, velocidad o tipo de calle.

En este caso, el parámetro `network_type="drive"` indica que solo se descargarán las vías **transitables por vehículos**, excluyendo senderos peatonales o ciclovías.

El resultado (`G`) es un objeto tipo `networkx.MultiDiGraph` que puede ser usado para:
- Calcular rutas más cortas entre puntos reales.
- Obtener longitudes, tiempos de recorrido o conexiones entre calles.
- Visualizar la malla vial completa o secciones específicas de la ciudad.

Para más detalles y opciones de configuración de OSMnx, puedes consultar la documentación oficial:  
[https://osmnx.readthedocs.io](https://osmnx.readthedocs.io)


In [ ]:
# Descargar la red vial para vehículos
G = ox.graph_from_place("Bogotá, Colombia", network_type="drive")

###  Visualización de la red vial

El siguiente comando utiliza la función `ox.plot_graph()` para representar gráficamente la red vial descargada desde **OpenStreetMap**.  
Cada línea gris corresponde a un tramo de vía y los nodos representan intersecciones o puntos de conexión entre calles.

El parámetro `node_size=0` oculta los nodos (para una visualización más limpia) y `edge_color='gray'` define el color de las vías.  
Esta visualización es útil para verificar que la red se haya descargado correctamente y observar la densidad del grafo vial en la zona de estudio.

In [ ]:
ox.plot_graph(G, node_size=0, edge_color='gray')

### Definición del depósito y generación de puntos de entrega

En esta celda se define el **depósito principal** (punto de partida y llegada de los vehículos), geocodificado con `ox.geocode()` a partir de una dirección real: *Zona Franca Fontibón, Bogotá*.  

Posteriormente, se generan **puntos de entrega aleatorios** dentro de los límites urbanos de Bogotá usando coordenadas aleatorias basadas en los valores mínimos y máximos del polígono de la ciudad.  
Estos puntos representarán los **clientes o destinos** que deben ser atendidos en el problema de ruteo.


In [ ]:
# Definir el depósito (por ejemplo, Zona Franca Fontibón)
depot_address = "Zona Franca Fontibón, Bogotá"
depot_coords = ox.geocode(depot_address)
cantidad_puntos = 10
random.seed(43)  # para reproducibilidad
# Generar puntos aleatorios dentro de Bogotá
area = ox.geocode_to_gdf("Bogotá, Colombia")
minx, miny, maxx, maxy = area.total_bounds
points = [(random.uniform(miny, maxy), random.uniform(minx, maxx)) for _ in range(cantidad_puntos)]

In [ ]:
# Convertir la lista de puntos a DataFrame
df_points = pd.DataFrame(points, columns=["Latitud", "Longitud"])

# Agregar un identificador para cada cliente
df_points.index = [f"Cliente_{i+1}" for i in range(len(df_points))]

# Mostrar el DataFrame ordenado
df_points

In [ ]:
# 1) Área de Bogotá y polígono (WGS84)
area = ox.geocode_to_gdf("Bogotá, Colombia").to_crs(4326)
poly = area.geometry.iloc[0]

# 2) Depósito (lat, lon)
depot_address = "Zona Franca Fontibón, Bogotá"
depot_latlon = ox.geocode(depot_address)  # -> (lat, lon)

# 3) Puntos aleatorios DENTRO del polígono (no solo en bounding box)
def random_point_in_polygon(polygon):
    minx, miny, maxx, maxy = polygon.bounds
    while True:
        p = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        if polygon.contains(p):
            # shapely: x=lon, y=lat  -> devolvemos (lat, lon)
            return (p.y, p.x)

num_clientes = 12
clientes = [random_point_in_polygon(poly) for _ in range(num_clientes)]  # lista de (lat, lon)

# 4) Mapa base
m = folium.Map(location=depot_latlon, zoom_start=12, tiles="CartoDB positron")

# (Opcional) dibujar límite de Bogotá
folium.GeoJson(
    data=area.__geo_interface__,
    name="Límite Bogotá",
    style_function=lambda x: {"fillColor": "#00000000", "color": "#666", "weight": 1},
).add_to(m)

# 5) Marcadores
folium.Marker(
    depot_latlon, tooltip="Depósito", icon=folium.Icon(color="red", icon="home")
).add_to(m)

for i, (lat, lon) in enumerate(clientes, start=1):
    folium.Marker(
        [lat, lon],
        tooltip=f"Cliente {i}",
        icon=folium.Icon(color="blue")
    ).add_to(m)

folium.LayerControl().add_to(m)

# 6) Mostrar en notebook y guardar como HTML (útil para compartir)
m.save("mapa_puntos_bogota.html")
m

In [ ]:
# --- Asignación de nodos más cercanos en la red vial ---

# Cada punto (depósito y clientes) debe asociarse al nodo más cercano del grafo vial (G)
# Esto permite que las rutas se calculen sobre la red real de calles, no por distancia euclidiana.

# 1️ El primer elemento corresponde al depósito (Zona Franca Fontibón)
#    ox.distance.nearest_nodes() busca el nodo de OSM más próximo a sus coordenadas (lon, lat)
# 2️ Luego, para cada punto de entrega aleatorio, se repite el proceso y se agregan al listado
# 3️ El resultado es una lista de identificadores de nodos (IDs únicos dentro del grafo)

nodos = [ox.distance.nearest_nodes(G, depot_coords[1], depot_coords[0])] + [
    ox.distance.nearest_nodes(G, lon, lat) for lat, lon in points
]

# Mostrar el listado de nodos asignados (1 depósito + n clientes)
print("Nodos asignados:", nodos)

### Cálculo de la matriz de distancias reales

En esta celda se construye la **matriz de distancias** entre todos los puntos (depósito y clientes), usando la red vial real representada en el grafo `G`.

Cada elemento `dist_matrix[i, j]` corresponde a la **distancia mínima en kilómetros** entre el nodo `i` y el nodo `j`, calculada mediante la función  
`nx.shortest_path_length()` de **NetworkX**, que evalúa la ruta más corta considerando el peso `'length'` (longitud real del tramo vial).

Si no existe conexión entre dos nodos, se asigna un valor de `np.inf` para indicar una distancia infinita.  
Esta matriz servirá como base para el **algoritmo genético** que buscará optimizar la ruta total del vehículo.


In [ ]:
# Matriz de distancias
dist_matrix = np.zeros((len(nodos), len(nodos)))
for i in range(len(nodos)):
    for j in range(len(nodos)):
        if i != j:
            try:
                dist_matrix[i, j] = nx.shortest_path_length(G, nodos[i], nodos[j], weight='length') / 1000
            except:
                dist_matrix[i, j] = np.inf  # si no hay conexión
print("Matriz de distancias (km):")

In [ ]:
# --- Visualización de la matriz de distancias como mapa de calor ---
import seaborn as sns

# Crear etiquetas para los nodos (Depósito + Clientes)
labels = ["Depósito"] + [f"Cliente_{i+1}" for i in range(len(nodos)-1)]

# Convertir la matriz a DataFrame para facilitar la visualización
df_dist = pd.DataFrame(dist_matrix, columns=labels, index=labels)

# Crear el mapa de calor
plt.figure(figsize=(8, 6))
sns.heatmap(df_dist, cmap="YlGnBu", annot=True, fmt=".2f", cbar_kws={"label": "Distancia (km)"})
plt.title("Matriz de Distancias Reales entre Nodos - Bogotá")
plt.xlabel("Destino")
plt.ylabel("Origen")
plt.tight_layout()
plt.show()


### Función de evaluación de la ruta

Esta función calcula la **distancia total** recorrida por un vehículo siguiendo un orden específico de visita (`ruta`).  
Suma las distancias entre nodos consecutivos usando la matriz `dist_matrix` y, al final, agrega el retorno al depósito (nodo 0).  
El valor devuelto representa el **fitness** que el algoritmo genético buscará minimizar.


In [ ]:
def cal_total_distance(ruta):
    total = 0
    for i in range(len(ruta) - 1):
        total += dist_matrix[int(ruta[i]), int(ruta[i+1])]
    # volver al depósito
    total += dist_matrix[int(ruta[-1]), 0]
    return total

### 🧬 Configuración del Algoritmo Genético (GA)

En esta celda se crea una instancia del algoritmo genético `GA_TSP` de la librería **scikit-opt**, diseñado para resolver el **Problema del Viajero (TSP)**, que es la base del ruteo de vehículos.

Los parámetros principales son:

- **`func=cal_total_distance`** → función objetivo que evalúa la distancia total de una ruta.  
- **`n_dim=len(nodos)`** → número de puntos a visitar (depósito + clientes).  
- **`size_pop=100`** → tamaño de la población, es decir, cuántas rutas posibles se evalúan por generación.  
- **`max_iter=200`** → número máximo de generaciones o iteraciones evolutivas.  
- **`prob_mut=0.2`** → probabilidad de mutación (20%), que introduce variaciones aleatorias para explorar nuevas soluciones.

Este proceso evolutivo imita la **selección natural**, permitiendo encontrar rutas cada vez más cortas a medida que avanzan las generaciones.


In [ ]:
ga_tsp = GA_TSP(
    func=cal_total_distance,
    n_dim=len(nodos),
    size_pop=100,
    max_iter=200,
    prob_mut=0.2
)

In [ ]:
# --- Ejecución del algoritmo genético y resultados finales ---

# Se ejecuta el proceso evolutivo completo del algoritmo genético definido previamente.
# Durante las iteraciones, el GA genera múltiples rutas posibles (individuos),
# calcula su distancia total usando la función de evaluación y selecciona las mejores
# combinaciones para la siguiente generación, aplicando operadores de cruce y mutación.

# Al finalizar, el método .run() devuelve:
#   - best_points: el mejor individuo encontrado (orden óptimo de visita de los nodos)
#   - best_distance: la distancia total mínima obtenida por el modelo

best_points, best_distance = ga_tsp.run()

# Mostrar los resultados del proceso de optimización
print(f"Mejor distancia total: {best_distance} km")
print("Mejor ruta (índices de nodos):", best_points)


In [ ]:
# --- Evolución del fitness (una muestra por generación) ---
fitness_history = history_scores if 'history_scores' in locals() else ga_tsp.all_history_Y

# Reducir la longitud a la mitad si hay duplicados
if len(fitness_history) > ga_tsp.max_iter:
    fitness_history = fitness_history[::2]

plt.figure(figsize=(8, 4))
plt.plot(fitness_history, color='blue', lw=2)
plt.title("Evolución del Fitness - GA_TSP")
plt.xlabel("Generación")
plt.ylabel("Distancia mínima (km)")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# --- Visualización de la mejor ruta obtenida con scikit-opt (Folium + OSMNX) ---

# 1. Obtener el orden óptimo de visita (excepto el depósito)


# 2. Convertir los índices en coordenadas reales
coords = [(G.nodes[nodos[i]]['y'], G.nodes[nodos[i]]['x']) for i in best_points]

# Asegurar que empieza y termina en el depósito
coords = [depot_coords] + coords + [depot_coords]

# 3. Crear el mapa
m2 = folium.Map(location=depot_coords, zoom_start=12, tiles="CartoDB positron")

# Marcador del depósito
folium.Marker(
    depot_coords,
    icon=folium.Icon(color='red', icon='home'),
    tooltip='Depósito'
).add_to(m2)

# Marcadores de clientes
for i, (lat, lon) in enumerate(coords[1:-1]):
    folium.Marker(
        [lat, lon],
        tooltip=f'Cliente {i+1}',
        icon=folium.Icon(color='blue')
    ).add_to(m2)

# 4. Dibujar la ruta óptima
folium.PolyLine(coords, color='blue', weight=4, opacity=0.8).add_to(m2)

# 5. Mostrar en el notebook
m2

# (Opcional) guardar el mapa como archivo HTML para abrirlo fuera de Jupyter
m2.save("ruta_optima_bogota.html")



In [ ]:
# --- Visualización correcta: ruta sobre calles + puntos alineados al grafo OSM ---

import folium
import networkx as nx
import numpy as np

depot_lat, depot_lon = depot_coords

# Convertir los índices de la mejor solución en nodos reales del grafo
best_points_ = np.array(best_points, dtype=int).flatten()
route_nodes = [nodos[0]] + [nodos[i] for i in best_points_] + [nodos[0]]

# Calcular la ruta real sobre el grafo (siguiendo las calles)
full_route = []
for i in range(len(route_nodes) - 1):
    try:
        path_segment = nx.shortest_path(G, route_nodes[i], route_nodes[i + 1], weight="length")
        full_route.extend(path_segment)
    except Exception as e:
        print(f"⚠️ No se pudo calcular el tramo entre {route_nodes[i]} y {route_nodes[i + 1]}: {e}")

# Obtener coordenadas de toda la ruta combinada
route_coords = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in full_route]

# Crear el mapa centrado en el depósito
m3 = folium.Map(location=depot_coords, zoom_start=12, tiles="CartoDB positron")

# Marcador del depósito
folium.Marker(
    depot_coords,
    icon=folium.Icon(color='red', icon='home'),
    tooltip='Depósito'
).add_to(m3)

# Marcadores de clientes
for i, (lat, lon) in enumerate(coords[1:-1]):
    folium.Marker(
        [lat, lon],
        tooltip=f'Cliente {i+1}',
        icon=folium.Icon(color='blue')
    ).add_to(m3)

# Dibujar la ruta óptima real (siguiendo calles)
folium.PolyLine(route_coords, color='blue', weight=4, opacity=0.8, tooltip="Ruta óptima real").add_to(m3)

# Mostrar mapa interactivo
m3

# Guardar el mapa
m3.save("ruta_optima_bogota_osm.html")
print("✅ Mapa guardado como 'ruta_optima_bogota_osm.html'")


